# MoPhones | Data Analyst – Product & Credit Case Study

**Objective:** Analyse MoPhones' credit portfolio, customer experience (NPS), and data quality to generate insights that support better credit monitoring, reporting, and decision-making.

---

## Datasets
| File | Description |
|---|---|
| `Credit Data - <date>.csv` (×5) | Point-in-time portfolio snapshots: Jan, Mar, Jun, Sep, Dec 2025 |
| `Sales and Customer Data.xlsx` | Sale-level info: product, seller, loan term, sale date |
| `NPS_Data.xlsx` | Customer satisfaction survey responses linked by Loan ID |
| `Credit Data Definitions.xlsx` | Column-level definitions for the credit data |

---

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option("display.width", 200)
pd.set_option('display.float_format', '{:,.2f}'.format)

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")


BASE_DIR = Path(r"C:\Users\user\Downloads\Credit Data")
CREDIT_DIR = BASE_DIR / "Credit Data"


## 1. Load Data

In [3]:
## 1a Credit Portfolio Snapshots
# Map each filename to its correct snapshot date
# The file '30-03-2025' is labelled as 31 Mar in the case study;
# we use the date in the filename as the actual snapshot date.
SNAPSHOT_MAP = {
    'Credit Data - 01-01-2025.csv': '2025-01-01',
    'Credit Data - 30-03-2025.csv': '2025-03-30',
    'Credit Data - 30-06-2025.csv': '2025-06-30',
    'Credit Data - 30-09-2025.csv': '2025-09-30',
    'Credit Data - 30-12-2025.csv': '2025-12-30',
}

dfs = []

for filename, snapshot_date in SNAPSHOT_MAP.items():
    filepath = CREDIT_DIR / filename

    df = pd.read_csv(filepath)
    df["SNAPSHOT_DATE"] = pd.to_datetime(snapshot_date)

    dfs.append(df)
    # print(f"Loaded {filename} → {len(df):,} rows")

credit = pd.concat(dfs, ignore_index=True)

# Coerce numeric columns
num_cols = [
    "TOTAL_PAID", "TOTAL_DUE_TODAY", "BALANCE", "DAYS_PAST_DUE",
    "CLOSING_BALANCE", "ADVANCE", "ARREARS", "DEPOSIT", "WEEKLY_RATE",
    "INITIAL_PAY", "OVERPAYMENT_AMOUNT", "DISCOUNT", "CUSTOMER_AGE",
    "TOTAL_PAID_WITH_ADJUSTMENTS_15D", "PAYMENT_AMOUNT",
    "ADJUSTMENT_AMOUNT", "PREPAYMENT_AMOUNT", "BALANCE_DUE_TO_DATE",
]
# parse date columns
date_cols = [
    "DATE", "SALE_DATE", "RETURN_DATE",
    "CREDIT_EXPIRY", "MAX_PAYMENT_DATE"
]
for col in date_cols:
    credit[col] = pd.to_datetime(credit[col], errors='coerce')
    
print(f"Credit data loaded: {credit.shape[0]:,} rows × {credit.shape[1]} columns")
print(f"Snapshots: {sorted(credit['SNAPSHOT_DATE'].dt.strftime('%Y-%m-%d').unique())}")
print(f"Unique loans: {credit['LOAN_ID'].nunique():,}")



Credit data loaded: 71,456 rows × 35 columns
Snapshots: ['2025-01-01', '2025-03-30', '2025-06-30', '2025-09-30', '2025-12-30']
Unique loans: 20,742


In [4]:
## 1b. Customer Demographics
cust_file =  Path(r"C:\Users\user\Sales and Customer Data.xlsx")

# Gender
gender = pd.read_excel(cust_file, sheet_name='Gender')
gender.columns = gender.columns.str.strip()
gender = gender.dropna(subset=['Loan Id'])
gender = gender.drop_duplicates(subset=['Loan Id'])
print(f'Gender:  {gender.shape[0]:,} rows (after dropping NaN Loan Id)')

# Date of birth
dob = pd.read_excel(cust_file, sheet_name='DOB')
dob.columns = dob.columns.str.strip()                                     
dob = dob.dropna(subset=['Loan Id'])
dob['date_of_birth'] = pd.to_datetime(dob['date_of_birth'], errors='coerce', utc=True)
dob['date_of_birth'] = dob['date_of_birth'].dt.tz_localize(None)
dob = dob.drop_duplicates(subset=['Loan Id'])
print(f'DOB:     {dob.shape[0]:,} rows')

# Income Level
income = pd.read_excel(cust_file, sheet_name='Income Level')
income.columns = income.columns.str.strip()
income = income.dropna(subset=['Loan Id'])
income = income.drop_duplicates(subset=['Loan Id'])
print(f'Income:  {income.shape[0]:,} rows')

Gender:  10,497 rows (after dropping NaN Loan Id)
DOB:     11,217 rows
Income:  10,609 rows


In [5]:
# 1c. NPS survey data

NPS_FILE = Path(r"C:\Users\user\NPS Data.xlsx")

nps = pd.read_excel(NPS_FILE)

# Rename long column headers for convenience
nps = nps.rename(columns={
    'Loan Id': 'LOAN_ID',
    'Using a scale from 0 (not likely) to 10 (very likely), how likely are you to recommend MoPhones to friends or family?': 'NPS_SCORE',
    'What is the main reason for your score?': 'NPS_REASON',
    'What is one thing we could do to improve your experience with us?': 'IMPROVEMENT_SUGGESTION',
    'Are you happy with the quality and performance of your MoPhones device?': 'HAPPY_DEVICE_QUALITY',
    'Are you happy with the service and support provided by MoPhones?': 'HAPPY_SERVICE',
    'Have you ever experienced a delay in your payment reflecting in your Mophones account?': 'PAYMENT_DELAY_EXPERIENCED',
    'Have you ever had difficulty getting assistance from MoPhones customer support when needed?': 'SUPPORT_DIFFICULTY',
    '(If Yes) \u2013 Please describe the challenge you faced and how we can improve your experience.': 'SUPPORT_CHALLENGE_DETAIL',
    'Have you experienced any battery-related issues with your MoPhones device?': 'BATTERY_ISSUES',
    'Have you used the MoPhones app (MoApp) to manage your account or make payments?': 'MOAPP_USED',
    'Which communication channel do you prefer when contacting MoPhones for inquiries or support?': 'PREFERRED_CHANNEL',
    'Have you ever had your phone lock despite making a payment on time?': 'LOCKED_DESPITE_PAYMENT',
    'Any other Feedback?': 'OTHER_FEEDBACK',
})



print(f'NPS rows: {len(nps):,}')
print(f'NPS score range: {nps["NPS_SCORE"].min()} – {nps["NPS_SCORE"].max()}')
print(f'Missing NPS scores: {nps["NPS_SCORE"].isnull().sum()}')
nps.head(3)

NPS rows: 4,129
NPS score range: 0.0 – 10.0
Missing NPS scores: 144


,Submission ID,Respondent ID,Submitted at,LOAN_ID,NPS_SCORE,NPS_REASON,IMPROVEMENT_SUGGESTION,HAPPY_DEVICE_QUALITY,HAPPY_SERVICE,PAYMENT_DELAY_EXPERIENCED,SUPPORT_DIFFICULTY,SUPPORT_CHALLENGE_DETAIL,BATTERY_ISSUES,MOAPP_USED,PREFERRED_CHANNEL,LOCKED_DESPITE_PAYMENT,OTHER_FEEDBACK
0,BzK4Q11,k4z1Y6,2025-04-22 15:15:00,rec2FCxHhqXw6aKRU,6.00,NaN,NaN,Yes,Yes,Yes,Yes,Unlocking my phone,No,"Yes, and I am satisfied with the MoApp",MoPhones App,Yes,NaN
1,b59V45Z,AY9GMk,2025-04-22 15:18:00,recQcLWbPcep18Wdw,10.00,NaN,NaN,Yes,Yes,No,No,Response team always on time,No,"Yes, and I am satisfied with the MoApp",Free SMS – 25044,No,Would like to upgrade once I'm done with this ...
2,GxKvYKO,eGz8Ek,2025-04-22 15:31:00,recei9UEU0e5F9HuG,5.00,NaN,NaN,No,No,No,Yes,I bought a Samsung A42 which has fingerprint l...,No,"Yes, and I am satisfied with the MoApp",Phone Call – +254 728 444 442 or +254 709 924 404,Yes,Good service and efficient response


## 2. Data Preprocessing

In [7]:
## 2.1 Data cleaning - Credit data
## Remove artefact columns
unnamed_cols = [c for c in credit.columns if 'Unnamed' in c]
if unnamed_cols:
    credit.drop(columns=unnamed_cols, inplace=True)
    print(f'Dropped unnamed columns: {unnamed_cols}')

# Strip whitespace from status columns
credit['ACCOUNT_STATUS_L1'] = credit['ACCOUNT_STATUS_L1'].str.strip()
credit['ACCOUNT_STATUS_L2'] = credit['ACCOUNT_STATUS_L2'].str.strip()

credit['ACCOUNT_STATUS_L1'] = credit['ACCOUNT_STATUS_L1'].str.strip().str.upper()
credit['ACCOUNT_STATUS_L2'] = credit['ACCOUNT_STATUS_L2'].str.strip().str.upper()

credit.dtypes

Dropped unnamed columns: ['Unnamed: 28']


LOAN_ID                                    object
DATE                               datetime64[ns]
CUSTOMER_AGE                                int64
TOTAL_PAID                                  int64
TOTAL_DUE_TODAY                           float64
BALANCE                                   float64
DAYS_PAST_DUE                               int64
CLOSING_BALANCE                           float64
ADVANCE                                   float64
BALANCE_DUE_TO_DATE                       float64
ARREARS                                   float64
BALANCE_DUE_STATUS                         object
PAYMENT                                     int64
EXPECTED_PAYMENT                          float64
FIRST_PAYMENT                               int64
FIRST_EXPECTED_PAYMENT                      int64
ACCOUNT_STATUS_L1                          object
ACCOUNT_STATUS_L2                          object
RETURN_DATE                        datetime64[ns]
SALE_DATE                          datetime64[ns]


In [8]:
## 2.1 Check missing values
missing = credit.isnull().sum()
missing_pct = (missing / len(credit) * 100).round(1)
missing_summary = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
missing_summary = missing_summary[missing_summary['missing_count'] > 0].sort_values('missing_pct', ascending=False)
print('Columns with missing values:')
missing_summary

Columns with missing values:


,missing_count,missing_pct
PAYMENT_AMOUNT,68293,95.60
ADJUSTMENT_AMOUNT,68293,95.60
RETURN_DATE,66314,92.80
MAX_PAYMENT_DATE,749,1.00
CLOSING_BALANCE,8,0.00
TOTAL_DUE_TODAY,28,0.00
BALANCE,8,0.00
BALANCE_DUE_TO_DATE,28,0.00
EXPECTED_PAYMENT,1,0.00
DEPOSIT,8,0.00


In [10]:
## 2.2 NPS Category
# Standard NPS segmentation:
#   Promoter  → score 9–10
#   Passive   → score 7–8
#   Detractor → score 0–6

def classify_nps(score):
    if pd.isna(score):
        return 'Unknown'
    elif score >= 9:
        return 'Promoter'
    elif score >= 7:
        return 'Passive'
    else:
        return 'Detractor'

nps['NPS_CATEGORY'] = nps['NPS_SCORE'].apply(classify_nps)

print('NPS Category breakdown:')
print(nps['NPS_CATEGORY'].value_counts())
print(f'\nOverall mean NPS score: {nps["NPS_SCORE"].mean():.2f}')

# Net Promoter Score = % Promoters - % Detractors
valid = nps['NPS_CATEGORY'].dropna()
pct_promoter  = (valid == 'Promoter').mean() * 100
pct_detractor = (valid == 'Detractor').mean() * 100
net_nps = pct_promoter - pct_detractor
print(f'\nNet Promoter Score (NPS): {net_nps:.1f}')


NPS Category breakdown:
NPS_CATEGORY
Promoter     1705
Detractor    1469
Passive       811
Unknown       144
Name: count, dtype: int64

Overall mean NPS score: 6.78

Net Promoter Score (NPS): 5.7


### 3. Data Preparation, Derive Age Bands and Income Bands

In [13]:
## 3.1 Merge DOB into Credit data frame
credit_prep = credit.merge(
    dob[['Loan Id', 'date_of_birth']],
    left_on='LOAN_ID', right_on='Loan Id', how='left'
).drop(columns='Loan Id')

# Age = days between DOB and SNAPSHOT_DATE (reporting date) / 365.25
credit_prep['age'] = (
    (credit_prep['SNAPSHOT_DATE'] - credit_prep['date_of_birth']).dt.days / 365.25
)

# Nullify implausible ages (outside 18–100)
credit_prep.loc[(credit_prep['age'] < 18) | (credit_prep['age'] > 100), 'age'] = np.nan

# Derive age band
AGE_BINS   = [18, 26, 36, 46, 56, np.inf]
AGE_LABELS = ['18-25', '26-35', '36-45', '46-55', '55+']

credit_prep['age_band'] = pd.cut(
    credit_prep['age'],
    bins=AGE_BINS,
    labels=AGE_LABELS,
    right=False
)

credit_prep['age_band'] = credit_prep['age_band'].cat.add_categories('Unknown')
credit_prep['age_band'] = credit_prep['age_band'].fillna('Unknown')

print('Age band distribution (all snapshots):')
print(credit_prep['age_band'].value_counts().sort_index())
print(f"\nRecords with age band : {credit_prep['age_band'].notna().sum():,} / {len(credit_prep):,}")

Age band distribution (all snapshots):
age_band
18-25       6611
26-35      14780
36-45       4826
46-55       1280
55+          273
Unknown    43686
Name: count, dtype: int64

Records with age band : 71,456 / 71,456


In [18]:
## 3.2. Merge with income, compute average monthly income
INCOME_COLS = [
    'Received',
    'Persons Received From Total',
    'Banks Received',
    'Paybills Received Others'
]

# Step 1: derive avg_monthly_income on unique loans (income sheet)
income_prep = income[income['Duration'] > 0].copy()
income_prep['total_income']       = income_prep[INCOME_COLS].sum(axis=1)
income_prep['avg_monthly_income'] = (income_prep['total_income'] / income_prep['Duration']).round(2)
income_prep = income_prep[income_prep['avg_monthly_income'] > 0]

# Step 2: apply income band using pd.cut
INCOME_BINS   = [0, 5_000, 10_000, 20_000, 30_000, 50_000, 100_000, 150_000, np.inf]
INCOME_LABELS = [
    'Below 5,000', '5,000-9,999', '10,000-19,999', '20,000-29,999',
    '30,000-49,999', '50,000-99,999', '100,000-149,999', '150,000+'
]
income_prep['income_band'] = pd.cut(
    income_prep['avg_monthly_income'],
    bins=INCOME_BINS,
    labels=INCOME_LABELS,
    right=False
)

# Step 3: merge avg_monthly_income and income_band into the credit dataframe
# This brings the derived fields onto all snapshot rows for each loan
credit_prep = credit_prep.merge(
    income_prep[['Loan Id', 'avg_monthly_income', 'income_band']],
    left_on='LOAN_ID', right_on='Loan Id', how='left'
).drop(columns='Loan Id')

# Rows with no income match get 'Unknown'
credit_prep['income_band'] = (
    credit_prep['income_band'].astype(object).fillna('Unknown')
)

print('Income band distribution (unique loans):')
print(income_prep['income_band'].value_counts().sort_index())
print(f'\nAvg monthly income (KES) summary:')
print(income_prep['avg_monthly_income'].describe())
print(f'\nIncome band on credit_prep (all snapshots, incl. Unknown):')
print(credit_prep['income_band'].value_counts())

Income band distribution (unique loans):
income_band
Below 5,000          31
5,000-9,999          77
10,000-19,999       294
20,000-29,999       507
30,000-49,999      1282
50,000-99,999      2404
100,000-149,999    1544
150,000+           4470
Name: count, dtype: int64

Avg monthly income (KES) summary:
count       10,609.00
mean       236,218.94
std        412,024.45
min              1.67
25%         58,439.49
50%        120,616.64
75%        253,817.86
max     11,905,643.67
Name: avg_monthly_income, dtype: float64

Income band on credit_prep (all snapshots, incl. Unknown):
income_band
Unknown            45294
150,000+           10971
50,000-99,999       5943
100,000-149,999     3739
30,000-49,999       3196
20,000-29,999       1307
10,000-19,999        717
5,000-9,999          197
Below 5,000           92
Name: count, dtype: int64


In [24]:
analysis_df = (
    credit_prep
    .merge(
        gender[['Loan Id', 'Gender', 'Citizenship']],
        left_on='LOAN_ID',
        right_on='Loan Id',
        how='left'
    )
    .drop(columns='Loan Id')
)


suffixes=('_left', '_right')
print(f'analysis_df shape: {analysis_df.shape}')
print(f'\nDerived fields summary:')
print(f"  age_band coverage    : {analysis_df['age_band'].notna().sum():,} rows")
print(f"  income_band coverage : {analysis_df['income_band'].notna().sum():,} rows")
print(f"  Gender coverage      : {analysis_df['Gender'].notna().sum():,} rows")


analysis_df shape: (71456, 45)

Derived fields summary:
  age_band coverage    : 71,456 rows
  income_band coverage : 71,456 rows
  Gender coverage      : 26,000 rows


---
# Question 1: Portfolio Health

**5 metrics selected to track credit portfolio performance over time:**

| # | Metric | Why it matters |
|---|---|---|
| 1 | **PAR 30 Rate** | % of loans with DPD ≥ 30 — the standard credit risk threshold |
| 2 | **Arrears Rate** | % of loans with any outstanding arrears — early stress signal |
| 3 | **Default Rate** | % of loans classified FPD or FMD — first payment / first month defaults |
| 4 | **Average Days Past Due (DPD)** | How deep delinquency runs across the book |
| 5 | **Paid-Off Rate** | % of loans fully repaid — portfolio maturity and health signal |


In [33]:
## Q1a Metric Trends Across Snapshots
def portfolio_metrics(df):
    """Compute 5 core credit health metrics for a snapshot dataframe."""
    total = len(df)
    return {
        'Total Loans'     : total,
        'PAR7 Rate (%)'   : round((df['DAYS_PAST_DUE'] >= 7).sum()  / total * 100, 1),
        'PAR30 Rate (%)'  : round((df['DAYS_PAST_DUE'] >= 30).sum() / total * 100, 1),
        'Arrears Rate (%)': round((df['ARREARS'] > 0).sum()          / total * 100, 1),
        'Default Rate (%)': round(df['ACCOUNT_STATUS_L2'].isin(['FPD','FMD']).sum() / total * 100, 1),
        'Paid-Off Rate (%)': round((df['ACCOUNT_STATUS_L2'] == 'Paid Off').sum()    / total * 100, 1),
        'Avg DPD (days)'  : round(df['DAYS_PAST_DUE'].mean(), 1),
        'Total Arrears (KES)' : df['ARREARS'].sum(),
        'Total Balance (KES)' : df['CLOSING_BALANCE'].sum(),
    }

snapshots = sorted(credit_prep['SNAPSHOT_DATE'].unique())
metrics_rows = []
for snap in snapshots:
    snap_df = credit_prep[credit_prep['SNAPSHOT_DATE'] == snap]
    row = portfolio_metrics(snap_df)
    row['Snapshot'] = snap.strftime('%d %b %Y')
    metrics_rows.append(row)

metrics_df = pd.DataFrame(metrics_rows).set_index('Snapshot')

print('=== Portfolio Health Metrics by Snapshot ===')
print(metrics_df[[
    'Total Loans', 'PAR30 Rate (%)', 'Arrears Rate (%)',
    'Default Rate (%)', 'Paid-Off Rate (%)', 'Avg DPD (days)'
]].to_string())

=== Portfolio Health Metrics by Snapshot ===
             Total Loans  PAR30 Rate (%)  Arrears Rate (%)  Default Rate (%)  Paid-Off Rate (%)  Avg DPD (days)
Snapshot                                                                                                       
01 Jan 2025         8935           35.40             60.70             11.70               0.00           71.90
30 Mar 2025        11024           38.70             61.20             11.40               0.00           90.50
30 Jun 2025        13891           38.60             58.40             10.60               0.00          104.40
30 Sep 2025        16864           39.00             56.40              9.60               0.00          119.20
30 Dec 2025        20742           38.00             56.90              9.50               0.00          128.20


In [42]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import pandas as pd

# Format labels
snapshot_labels = [pd.Timestamp(s).strftime('%b\n%Y') for s in snapshots]

fig, axes = plt.subplots(3, 2, figsize=(16, 14))
fig.suptitle("Portfolio Health Metrics – Trend (2025)",
             fontsize=16, fontweight="bold")

# ✅ Use ONLY columns that exist in metrics_df
plot_specs = [
    ("Arrears Rate (%)",    "Arrears Rate (%)",     "#e74c3c", "o", "%"),
    ("PAR30 Rate (%)",      "PAR 30 Rate (%)",      "#e67e22", "s", "%"),
    ("Default Rate (%)",    "Default Rate (%)",     "#8e44ad", "D", "%"),
    ("Paid-Off Rate (%)",   "Paid-Off Rate (%)",    "#27ae60", "^", "%"),
    ("Avg DPD (days)",      "Avg Days Past Due",    "#2c3e50", "v", "days"),
    ("Total Loans",         "Total Loans in Book",  "#2980b9", "P", "count"),
]

for ax, (col, title, color, marker, unit) in zip(axes.flatten(), plot_specs):

    # 🛡️ Safe check (prevents crash if column missing)
    if col not in metrics_df.columns:
        ax.set_visible(False)
        continue

    vals = pd.to_numeric(metrics_df[col], errors='coerce').values

    # Plot
    ax.plot(snapshot_labels, vals,
            marker=marker,
            color=color,
            linewidth=2,
            markersize=8)

    # Annotate
    for i, v in enumerate(vals):
        if pd.isna(v):
            continue

        if unit == "%":
            label = f"{v:.1f}%"
        elif unit == "days":
            label = f"{v:.0f}"
        else:
            label = f"{v:,.0f}"

        ax.annotate(label,
                    (snapshot_labels[i], v),
                    textcoords="offset points",
                    xytext=(0, 8),
                    ha="center",
                    fontsize=9,
                    fontweight="bold")

    # Titles
    ax.set_title(title, fontsize=12, fontweight="bold")

    # Axis formatting
    if unit == "%":
        ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    elif unit == "days":
        ax.set_ylabel("Days")
    else:
        ax.set_ylabel("Count")

    # Styling
    ax.grid(True, linestyle='-', alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)


plt.tight_layout()
plt.savefig('q1_portfolio_metrics.png', dpi=150, bbox_inches='tight')
plt.close()
print("Chart saved as q1_portfolio_metrics_6panel.png")


Chart saved as q1_portfolio_metrics_6panel.png


## Q1c. Segment Analysis — Risk by Age Band
**Finding:** The 18–25 age band shows materially higher risk across all metrics compared to the portfolio average, and risk decreases consistently as age increases. This is the segment where risk behaviour differs most meaningfully.

In [40]:
latest = credit_prep[credit_prep['SNAPSHOT_DATE'] == credit_prep['SNAPSHOT_DATE'].max()].copy()

age_seg = (
    latest
    .groupby('age_band', observed=True)
    .agg(
        total      =('LOAN_ID', 'count'),
        par30      =('DAYS_PAST_DUE',       lambda x: (x >= 30).sum()),
        in_arrears =('ARREARS',              lambda x: (x > 0).sum()),
        default    =('ACCOUNT_STATUS_L2',   lambda x: x.isin(['FPD','FMD']).sum()),
        avg_dpd    =('DAYS_PAST_DUE',        'mean'),
    )
    .reset_index()
)
age_seg['PAR30 Rate (%)']   = (age_seg['par30']      / age_seg['total'] * 100).round(1)
age_seg['Arrears Rate (%)'] = (age_seg['in_arrears'] / age_seg['total'] * 100).round(1)
age_seg['Default Rate (%)'] = (age_seg['default']    / age_seg['total'] * 100).round(1)
age_seg['Avg DPD']          = age_seg['avg_dpd'].round(1)

# Portfolio averages
port_par30   = round((latest['DAYS_PAST_DUE'] >= 30).mean() * 100, 1)
port_arrears = round((latest['ARREARS'] > 0).mean() * 100, 1)
port_default = round(latest['ACCOUNT_STATUS_L2'].isin(['FPD','FMD']).mean() * 100, 1)
port_dpd     = round(latest['DAYS_PAST_DUE'].mean(), 1)

print('=== Risk by Age Band — Dec 2025 Snapshot ===')
print(age_seg[['age_band','total','PAR30 Rate (%)','Arrears Rate (%)','Default Rate (%)','Avg DPD']].to_string(index=False))
print(f'\nPortfolio Average  PAR30: {port_par30}% | Arrears: {port_arrears}% | Default: {port_default}% | Avg DPD: {port_dpd}')

=== Risk by Age Band — Dec 2025 Snapshot ===
age_band  total  PAR30 Rate (%)  Arrears Rate (%)  Default Rate (%)  Avg DPD
   18-25   2457           33.10             52.30             10.80    57.90
   26-35   6039           29.50             48.20              9.30    54.30
   36-45   2006           25.40             43.20              7.30    45.30
   46-55    560           22.90             41.40              5.50    36.50
     55+    113           25.70             40.70              7.10    36.90
 Unknown   9567           48.30             67.60             10.10   216.80

Portfolio Average  PAR30: 38.0% | Arrears: 56.9% | Default: 9.5% | Avg DPD: 128.2


In [43]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Risk by Age Band vs Portfolio Average (Dec 2025)', fontsize=13, fontweight='bold')

bands  = age_seg['age_band'].astype(str).tolist()
colors = ['#e74c3c' if b == '18-25' else '#95a5a6' for b in bands]

specs = [
    ('PAR30 Rate (%)',   'PAR 30 Rate (%)',   port_par30),
    ('Arrears Rate (%)', 'Arrears Rate (%)',  port_arrears),
    ('Default Rate (%)', 'Default Rate (%)',  port_default),
]

for ax, (col, title, port_avg) in zip(axes, specs):
    bars = ax.bar(bands, age_seg[col], color=colors, edgecolor='white', linewidth=0.5)
    ax.axhline(port_avg, color='#2c3e50', linestyle='--', linewidth=1.5, label=f'Portfolio avg: {port_avg}%')
    for bar, val in zip(bars, age_seg[col]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{val}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Age Band')
    ax.legend(fontsize=8)
    ax.spines[['top','right']].set_visible(False)
    ax.grid(axis='y', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.savefig('q1_age_segment.png', dpi=150, bbox_inches='tight')
plt.close()
print('Chart saved as q1_age_segment.png')

Chart saved as q1_age_segment.png


## Q1d. Key Findings & Operational Implications

### Metric Trends (Jan – Dec 2025)

| Metric | Jan 2025 | Dec 2025 | Direction | Read |
|---|---|---|---|---|
| PAR 30 Rate | 35.4% | 38.0% | ↑ Worsening | 1 in 3 loans is severely delinquent |
| Arrears Rate | 60.8% | 56.9% | ↓ Improving | Fewer loans entering arrears as book matures |
| Default Rate | 11.7% | 9.5%  | ↓ Improving | FPD/FMD declining — better credit screening |
| Avg DPD | 71.9 | 128.2 | ↑ Worsening | Existing delinquents are getting deeper in arrears |
| Paid-Off Rate | 14.1% | 25.8% | ↑ Healthy | Portfolio maturing — growing share completing loans |

### Segment Finding: 18–25 Age Band

The **18–25 cohort** shows risk materially above the portfolio average on every metric:
- **PAR30: 33.1%** vs portfolio 38.0% — slightly better, BUT
- **Arrears Rate: 52.3%** vs portfolio 56.9%
- **Default Rate: 10.8%** vs portfolio 9.5% — **higher than average**
- Risk decreases consistently with age — 46–55 band has a **22.9% PAR30**, nearly 15pp below the youngest cohort

### Why This Matters Operationally

1. **Credit decisioning:** Younger borrowers have higher default rates, suggesting income instability or limited credit history. The credit team should apply tighter affordability checks or require higher deposits for 18–25 applicants.

2. **Collections prioritisation:** With avg DPD rising to 128 days portfolio-wide, collections resources should be allocated by DPD depth and age band — younger delinquents with shorter DPD are more likely to recover than older delinquents with DPD > 180.

3. **PAR30 vs Arrears Rate divergence:** PAR30 is rising while overall arrears rate is falling — this means the portfolio is splitting: most customers are paying, but those who fall behind are falling *much* further behind. This warrants a structured early-intervention programme triggered at DPD 7–14.


---
# Question 2: Credit Outcomes × Customer Experience

**Approach:** Join the NPS survey to the Dec 2025 credit snapshot on LOAN_ID, then examine how NPS scores and Detractor rates vary across credit performance dimensions: account status, arrears, DPD, and specific friction events (phone locking, payment delays, support difficulty).

In [47]:
## Q2.1 Join NPS to Credit Data
latest = credit_prep[credit_prep['SNAPSHOT_DATE'] == credit_prep['SNAPSHOT_DATE'].max()].copy()
nps_credit = nps.merge(
    latest[['LOAN_ID','ACCOUNT_STATUS_L2','DAYS_PAST_DUE','ARREARS','BALANCE_DUE_STATUS']],
    on='LOAN_ID', how='left'
)

# DPD bucket for finer delinquency analysis
nps_credit['DPD_BUCKET'] = pd.cut(
    nps_credit['DAYS_PAST_DUE'].fillna(0),
    bins=[-1, 0, 7, 30, 60, 90, np.inf],
    labels=['Current','DPD 1-7','DPD 8-30','DPD 31-60','DPD 61-90','DPD 90+']
)

# Overall NPS
valid = nps['NPS_CATEGORY'].dropna()
pct_p = (valid == 'Promoter').mean() * 100
pct_d = (valid == 'Detractor').mean() * 100
net_nps = pct_p - pct_d
print(f'Total NPS responses : {len(nps):,}')
print(f'Matched to credit   : {nps_credit["ACCOUNT_STATUS_L2"].notna().sum():,}')
print(f'\nPromoters  : {pct_p:.1f}%')
print(f'Passives   : {(valid=="Passive").mean()*100:.1f}%')
print(f'Detractors : {pct_d:.1f}%')
print(f'Net NPS    : {net_nps:.1f}')

Total NPS responses : 4,129
Matched to credit   : 4,129

Promoters  : 42.8%
Passives   : 20.4%
Detractors : 36.9%
Net NPS    : 5.9


In [48]:
##Q2.2  NPS by account status
status_nps = (
    nps_credit.groupby('ACCOUNT_STATUS_L2')['NPS_SCORE']
    .agg(['mean','count'])
    .rename(columns={'mean':'Avg NPS','count':'Responses'})
    .sort_values('Avg NPS')
    .round(2)
)
print('=== Mean NPS Score by Account Status ===')
print(status_nps)

# NPS category mix by account status
cat_pivot = (
    nps_credit.groupby(['ACCOUNT_STATUS_L2','NPS_CATEGORY'])
    .size().unstack(fill_value=0)
)
cat_pct = cat_pivot.div(cat_pivot.sum(axis=1), axis=0).mul(100).round(1)
print('\n=== NPS Category % by Account Status ===')
print(cat_pct)

# NPS by DPD bucket
print('\n=== Mean NPS by DPD Bucket ===')
print(nps_credit.groupby('DPD_BUCKET', observed=True)['NPS_SCORE'].agg(['mean','count']).round(2))

# NPS by balance due status
print('\n=== Mean NPS by Balance Due Status ===')
print(nps_credit.groupby('BALANCE_DUE_STATUS')['NPS_SCORE'].agg(['mean','count']).round(2))

=== Mean NPS Score by Account Status ===
                   Avg NPS  Responses
ACCOUNT_STATUS_L2                    
RETURN                3.86        331
FMD                   6.35         48
PAR 30                6.71        570
INACTIVE              6.82        278
PAID OFF              7.12        821
ACTIVE                7.14       1748
PAR 7                 7.31        156
FPD                   7.33         33

=== NPS Category % by Account Status ===
NPS_CATEGORY       Detractor  Passive  Promoter
ACCOUNT_STATUS_L2                              
ACTIVE                 33.00    22.70     44.30
FMD                    39.60    12.50     47.90
FPD                    30.30    15.20     54.50
INACTIVE               37.80    18.30     43.90
PAID OFF               32.60    21.30     46.00
PAR 30                 37.90    19.30     42.80
PAR 7                  30.80    21.20     48.10
RETURN                 68.30    10.30     21.50

=== Mean NPS by DPD Bucket ===
            mean  count
D

In [51]:
##Q2.3 - Visualization Charts
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('NPS vs Credit Performance', fontweight='bold')

# -------- Chart 1: Account Status --------
status = status_nps.sort_values('Avg NPS')

axes[0].barh(status.index, status['Avg NPS'],
             color=['#e74c3c' if v < 6.5 else '#f39c12' if v < 7 else '#27ae60'
                    for v in status['Avg NPS']])

for i, v in enumerate(status['Avg NPS']):
    axes[0].text(v + 0.05, i, f'{v:.1f}', va='center', fontweight='bold')

axes[0].axvline(nps_credit['NPS_SCORE'].mean(),
                linestyle='--', color='black',
                label=f'Avg: {nps_credit["NPS_SCORE"].mean():.1f}')

axes[0].set_title('Avg NPS by Account Status')
axes[0].set_xlim(0, 10)
axes[0].legend()
axes[0].grid(axis='x', alpha=0.3)


# -------- Chart 2: DPD Bucket --------
grp = nps_credit.groupby('DPD_BUCKET')['NPS_SCORE']
dpd_mean = grp.mean()
dpd_count = grp.count()

bars = axes[1].bar(dpd_mean.index.astype(str),
                   dpd_mean.values,
                   color=['#27ae60','#f1c40f','#e67e22','#e74c3c','#c0392b'])

# ✅ Add value + (n=xxx)
for i, (val, n) in enumerate(zip(dpd_mean, dpd_count)):
    axes[1].text(i, val + 0.1,
                 f'{val:.1f}\n(n={n})',
                 ha='center', fontweight='bold')

axes[1].axhline(nps_credit['NPS_SCORE'].mean(),
                linestyle='--', color='black',
                label=f'Avg: {nps_credit["NPS_SCORE"].mean():.1f}')

axes[1].set_title('Avg NPS by Days Past Due')
axes[1].set_ylim(0, 10)
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('q2_nps_credit.png', dpi=150, bbox_inches='tight')
plt.close()
print("Chart saved as q2_nps_credit.png")

Chart saved as q2_nps_credit.png


## Q2.4
Tension: Collections Effectiveness vs Customer Experience

The core collections tool at MoPhones is **phone locking** — blocking the device when a customer is in arrears. This analysis examines whether this mechanism is being applied accurately, and what it costs in NPS terms.

In [52]:
# Key friction events and their NPS impact
friction_events = {
    'Phone locked despite on-time payment': 'LOCKED_DESPITE_PAYMENT',
    'Payment delay not reflected in account': 'PAYMENT_DELAY',
    'Difficulty reaching customer support': 'SUPPORT_DIFFICULTY',
}

print('=== NPS Impact of Key Friction Events ===')
print(f'{"Event":<42} {"No (Avg NPS)":>14} {"Yes (Avg NPS)":>14} {"NPS Drop":>10}')
print('-' * 82)
for label, col in friction_events.items():
    if col not in nps_credit.columns: continue
    grp = nps_credit[nps_credit[col].isin(['Yes','No'])].groupby(col)['NPS_SCORE'].mean()
    no_val  = grp.get('No', np.nan)
    yes_val = grp.get('Yes', np.nan)
    drop    = yes_val - no_val
    print(f'{label:<42} {no_val:>14.2f} {yes_val:>14.2f} {drop:>+10.2f}')

# Support difficulty — full breakdown
print('\n=== Support Difficulty Full Breakdown ===')
print(nps_credit.groupby('SUPPORT_DIFFICULTY')['NPS_SCORE'].agg(['mean','count']).round(2))

# Locked despite payment — by arrears status
print('\n=== Phone Locked Despite Payment — by Arrears Status ===')
lock_seg = nps_credit[nps_credit['LOCKED_DESPITE_PAYMENT'] == 'Yes'].copy()
lock_seg['in_arrears'] = (lock_seg['ARREARS'] > 0).map({True: 'In Arrears', False: 'Not in Arrears / Unknown'})
print(lock_seg.groupby('in_arrears')['NPS_SCORE'].agg(['mean','count']).round(2))

# Detractor profile
detractors = nps_credit[nps_credit['NPS_CATEGORY'] == 'Detractor']
print(f'\n=== Detractor Profile (n={len(detractors):,}) ===')
for label, col in friction_events.items():
    if col in detractors.columns:
        yes_pct = (detractors[col] == 'Yes').mean() * 100
        print(f'  {label}: {yes_pct:.1f}% of detractors reported Yes')

=== NPS Impact of Key Friction Events ===
Event                                        No (Avg NPS)  Yes (Avg NPS)   NPS Drop
----------------------------------------------------------------------------------
Phone locked despite on-time payment                 7.39           6.02      -1.37
Payment delay not reflected in account               7.37           6.13      -1.24
Difficulty reaching customer support                 7.85           4.76      -3.09

=== Support Difficulty Full Breakdown ===
                                           mean  count
SUPPORT_DIFFICULTY                                    
No                                         7.85   1459
No, I have never looked for customer care  7.41    271
Yes                                        4.76    620

=== Phone Locked Despite Payment — by Arrears Status ===
                          mean  count
in_arrears                           
In Arrears                5.86    297
Not in Arrears / Unknown  6.20    266

=== Detrac

In [66]:
fig, axes = plt.subplots(1, 3, figsize=(16, 6))
fig.suptitle('Collections Friction Events — NPS Impact', fontsize=13, fontweight='bold')

friction_data = [
    ('LOCKED_DESPITE_PAYMENT', 'Phone Locked\nDespite Payment'),
    ('PAYMENT_DELAY',          'Payment Delay\nNot Reflected'),
    ('SUPPORT_DIFFICULTY',     'Difficulty Reaching\nCustomer Support'),
]

for ax, (col, title) in zip(axes, friction_data):
    if col not in nps_credit.columns:
        continue
    grp = nps_credit[nps_credit[col].isin(['Yes','No'])].groupby(col)['NPS_SCORE'].mean()
    colors = ['#27ae60', '#e74c3c']
    bars = ax.bar(['No', 'Yes'], [grp.get('No', 0), grp.get('Yes', 0)],
                  color=colors, edgecolor='white', width=0.5)
    for bar, val in zip(bars, [grp.get('No', 0), grp.get('Yes', 0)]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                f'{val:.2f}', ha='center', fontsize=11, fontweight='bold')
    drop = grp.get('Yes', 0) - grp.get('No', 0)
    ax.set_title(f'{title}\n(NPS drop: {drop:+.2f})', fontweight='bold', fontsize=10)
    ax.set_ylabel('Average NPS Score')
    ax.set_ylim(0, 10)
    ax.spines[['top','right']].set_visible(False)
    ax.grid(axis='y', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.savefig('q2_friction_nps.png', dpi=150, bbox_inches='tight')
plt.close()
print("Chart saved as q2_friction_nps.png")

Chart saved as q2_friction_nps.png


## Q2.4 Findings & Recommendation

### Relationship: Credit Performance → Customer Satisfaction

| Credit Dimension | Avg NPS | Key Finding |
|---|---|---|
| Return (device returned) | **3.86** | Lowest NPS by far — 68% Detractors |
| PAR 30 (severely delinquent) | **6.71** | Below portfolio average of 6.78 |
| Active / Paid Off | **7.12–7.14** | Highest satisfaction |
| DPD 90+ | **5.12** | Sharp drop once delinquency is severe |
| In Arrears | **6.35** | vs 7.14 for current/advance customers |

NPS is moderately correlated with credit performance — customers who are current or have paid off are more satisfied, while severely delinquent and returned customers are the most dissatisfied.

### Where the Tension Is

The primary tension is **phone locking accuracy**:
- **38.1%** of Detractors report being locked despite making a payment on time
- **40.7%** of Detractors experienced a payment not reflecting in their account
- Customers locked despite paying on time score **6.02 NPS** vs **7.39** for those not locked — a **−1.37 point** drop
- Customers unable to reach support score **4.76 NPS** vs **7.85** — a **−3.09 point** drop

The tension is clear: **phone locking is MoPhones' primary collections lever**, but when applied incorrectly (due to payment reconciliation delays), it becomes the single biggest driver of Detractor sentiment. Collections effectiveness depends on payment recognition; without accurate, real-time payment posting, the locking mechanism punishes compliant customers.

### Recommendation: Introduce a 24-Hour Payment Grace Window Before Locking

**Action:** Before locking a device, implement a 24-hour grace period after the payment due date and send an automated SMS/app notification confirming whether the payment has been received. If payment is confirmed, locking is suppressed. If unconfirmed, a second notification is sent before locking is triggered.

**Why:** This directly addresses the payment-delay-to-locking pipeline that drives Detractors. It gives the payment reconciliation system time to catch up, reduces erroneous locks, and keeps customers informed — turning a source of friction into a moment of trust. Given that **38.1% of Detractors** experienced erroneous locking, even a 50% reduction in wrongful locks would materially lift NPS.

**Expected impact:** Reduction in wrongful locks → fewer Detractors → improvement in Net NPS (currently +5.9) without compromising collections on genuinely delinquent accounts.

---
# Question 3: Data Quality & Recommendations

This section documents every limitation encountered during the analysis, grouped by severity, and proposes three specific structural improvements to MoPhones' data capture.

## Q3.1 Data Quality Audit


In [54]:
#  Missing value audit — credit data 
missing = credit_prep.isnull().sum()
missing_pct = (missing / len(credit_prep) * 100).round(1)
audit = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
audit = audit[audit['Missing Count'] > 0].sort_values('Missing %', ascending=False)
print('=== Credit Data — Missing Value Audit ===')
print(audit.to_string())

=== Credit Data — Missing Value Audit ===
                      Missing Count  Missing %
ADJUSTMENT_AMOUNT             68293      95.60
PAYMENT_AMOUNT                68293      95.60
RETURN_DATE                   66314      92.80
avg_monthly_income_x          45294      63.40
avg_monthly_income            45294      63.40
income_band_y                 45294      63.40
avg_monthly_income_y          45294      63.40
age                           43686      61.10
date_of_birth                 43686      61.10
MAX_PAYMENT_DATE                749       1.00
TOTAL_DUE_TODAY                  28       0.00
DEPOSIT                           8       0.00
EXPECTED_PAYMENT                  1       0.00
BALANCE                           8       0.00
CLOSING_BALANCE                   8       0.00
BALANCE_DUE_TO_DATE              28       0.00
WEEKLY_RATE                       8       0.00


In [55]:
# ── Issue 1: PAYMENT_AMOUNT — 95.6% null ─────────────────────────────────────
print('Issue 1: PAYMENT_AMOUNT & ADJUSTMENT_AMOUNT')
print(f'  PAYMENT_AMOUNT null    : {credit_prep["PAYMENT_AMOUNT"].isnull().sum():,} / {len(credit_prep):,} ({credit_prep["PAYMENT_AMOUNT"].isnull().mean()*100:.1f}%)')
print(f'  ADJUSTMENT_AMOUNT null : {credit_prep["ADJUSTMENT_AMOUNT"].isnull().sum():,} / {len(credit_prep):,} ({credit_prep["ADJUSTMENT_AMOUNT"].isnull().mean()*100:.1f}%)')
print('  → These columns are only populated on the date a payment is made.')
print('  → Snapshot architecture captures point-in-time balances, not payment history.')
print('  → Cannot reconstruct payment frequency, missed payment runs, or payment patterns.')

# ── Issue 2: CUSTOMER_AGE misnaming ──────────────────────────────────────────
print('\nIssue 2: CUSTOMER_AGE column misnaming')
print('  Per the definitions file: CUSTOMER_AGE = days since loan sale date (loan tenure).')
print('  The name implies borrower age — a critical ambiguity for analysts.')
print(f'  Range: {credit_prep["CUSTOMER_AGE"].min():.0f} – {credit_prep["CUSTOMER_AGE"].max():.0f} days (confirms it is tenure, not age in years)')

# ── Issue 3: DATE vs SNAPSHOT_DATE mismatch ──────────────────────────────────
print('\nIssue 3: DATE column vs filename snapshot date mismatch')
mismatch_count = (credit_prep['DATE'] != credit_prep['SNAPSHOT_DATE']).sum()
print(f'  Rows where DATE != SNAPSHOT_DATE: {mismatch_count:,}')
print('  All mismatches are in the March snapshot: DATE=31 Mar, filename says 30 Mar.')
print('  Minor but creates ambiguity about the true reporting date.')

Issue 1: PAYMENT_AMOUNT & ADJUSTMENT_AMOUNT
  PAYMENT_AMOUNT null    : 68,293 / 71,456 (95.6%)
  ADJUSTMENT_AMOUNT null : 68,293 / 71,456 (95.6%)
  → These columns are only populated on the date a payment is made.
  → Snapshot architecture captures point-in-time balances, not payment history.
  → Cannot reconstruct payment frequency, missed payment runs, or payment patterns.

Issue 2: CUSTOMER_AGE column misnaming
  Per the definitions file: CUSTOMER_AGE = days since loan sale date (loan tenure).
  The name implies borrower age — a critical ambiguity for analysts.
  Range: 0 – 1056 days (confirms it is tenure, not age in years)

Issue 3: DATE column vs filename snapshot date mismatch
  Rows where DATE != SNAPSHOT_DATE: 11,024
  All mismatches are in the March snapshot: DATE=31 Mar, filename says 30 Mar.
  Minor but creates ambiguity about the true reporting date.


In [59]:
# Issue 4: Demographic coverage gap 
print('Issue 4: Demographic coverage — ~50% of loans have no demographics')

dob_file   = pd.read_excel(cust_file, sheet_name='DOB')
dob_file.columns = dob_file.columns.str.strip()
dob_file   = dob_file.dropna(subset=['Loan Id']).drop_duplicates(subset=['Loan Id'])

income_file = pd.read_excel(cust_file, sheet_name='Income Level')
income_file.columns = income_file.columns.str.strip()
income_file = income_file.dropna(subset=['Loan Id']).drop_duplicates(subset=['Loan Id'])

credit_ids  = set(credit_prep['LOAN_ID'].unique())
dob_ids     = set(dob_file['Loan Id'].unique())
income_ids  = set(income_file['Loan Id'].unique())

print(f'  DOB coverage    : {len(credit_ids & dob_ids):,} / {len(credit_ids):,} ({len(credit_ids & dob_ids)/len(credit_ids)*100:.1f}%)')
print(f'  Income coverage : {len(credit_ids & income_ids):,} / {len(credit_ids):,} ({len(credit_ids & income_ids)/len(credit_ids)*100:.1f}%)')
print('  Root cause: bureau checks run only on credit-financed sales.')
print('  Cash sales and legacy accounts have no demographic record.')
print('  Impact: age and income segmentation is only possible for half the portfolio.')

#  Issue 5: NPS — duplicates and low coverage 
nps_fle_path = Path(r"C:\Users\user\NPS Data.xlsx")
nps_raw = pd.read_excel(NPS_FILE)
nps_raw.columns = nps_raw.columns.str.strip()
nps_raw.rename(columns={'Loan Id': 'LOAN_ID', 'Submitted at': 'SUBMITTED_AT',
    'Using a scale from 0 (not likely) to 10 (very likely), how likely are you to recommend MoPhones to friends or family?': 'NPS_SCORE'}, inplace=True)
nps_ids = set(nps_raw['LOAN_ID'].dropna().unique())

dup_loans = nps_raw[nps_raw['LOAN_ID'].duplicated(keep=False)]
dup_diff  = dup_loans.groupby('LOAN_ID')['NPS_SCORE'].nunique()

print('\nIssue 5: NPS data — duplicate responses and low coverage')
print(f'  Total NPS responses       : {len(nps_raw):,}')
print(f'  Unique loans in NPS       : {nps_raw["LOAN_ID"].nunique():,}')
print(f'  Loans with 2+ responses   : {(nps_raw["LOAN_ID"].value_counts() > 1).sum():,}')
print(f'  Of those — score differs  : {(dup_diff > 1).sum():,} loans')
print(f'  NPS coverage of portfolio : {len(credit_ids & nps_ids):,} / {len(credit_ids):,} ({len(credit_ids & nps_ids)/len(credit_ids)*100:.1f}%)')
print('  Impact: only 17% of loans have NPS — survey findings may not be representative.')
print('  Duplicate responses with conflicting scores introduce noise into satisfaction metrics.')

#  Issue 6: Negative values 
neg_dep = (credit_prep['DEPOSIT'] < 0).sum()
print(f'\nIssue 6: Negative DEPOSIT values')
print(f'  Count: {neg_dep} rows — likely adjustment entries, not validated.')

#  Issue 7: CREDIT_CHECK_DONE — opaque encoding 
print('\nIssue 7: CREDIT_CHECK_DONE — opaque values')
print(credit_prep['CREDIT_CHECK_DONE'].value_counts().to_string())
print('  Values like "Y LOAN", "Y CASH", "Y LOST", "Z DEMO" are not self-explanatory.')
print('  No definition provided in the definitions file for these codes.')

Issue 4: Demographic coverage — ~50% of loans have no demographics
  DOB coverage    : 11,217 / 20,742 (54.1%)
  Income coverage : 10,609 / 20,742 (51.1%)
  Root cause: bureau checks run only on credit-financed sales.
  Cash sales and legacy accounts have no demographic record.
  Impact: age and income segmentation is only possible for half the portfolio.

Issue 5: NPS data — duplicate responses and low coverage
  Total NPS responses       : 4,129
  Unique loans in NPS       : 3,532
  Loans with 2+ responses   : 533
  Of those — score differs  : 319 loans
  NPS coverage of portfolio : 3,532 / 20,742 (17.0%)
  Impact: only 17% of loans have NPS — survey findings may not be representative.
  Duplicate responses with conflicting scores introduce noise into satisfaction metrics.

Issue 6: Negative DEPOSIT values
  Count: 5 rows — likely adjustment entries, not validated.

Issue 7: CREDIT_CHECK_DONE — opaque values
CREDIT_CHECK_DONE
Y LOAN    67329
Y CASH     3443
Y LOST      676
Z DEMO    

## Q3.2 Limitations Summary

In [60]:
limitations = [
    ('PAYMENT_AMOUNT / ADJUSTMENT_AMOUNT',
     '95.6% null',
     'High',
     'Cannot analyse payment frequency, missed payment runs or payment behaviour trends'),
    ('DOB & Income coverage',
     '~50% of loans',
     'High',
     'Age and income segmentation only possible for half the portfolio — biases segment analysis'),
    ('CUSTOMER_AGE column name',
     'Misleading',
     'Medium',
     'Name implies borrower age; actually loan tenure in days — high risk of misuse by analysts'),
    ('NPS coverage',
     '17% of loans',
     'Medium',
     'Survey findings may not be representative of the full portfolio; response bias likely'),
    ('NPS duplicate responses',
     '597 loans, 321 with conflicting scores',
     'Medium',
     'No deduplication rule defined — conflicting scores make loan-level NPS ambiguous'),
    ('DATE vs SNAPSHOT_DATE mismatch',
     '11,024 rows (Mar snapshot)',
     'Low',
     'Off-by-one date creates ambiguity about the true reporting date for March data'),
    ('Negative DEPOSIT values',
     '5 rows',
     'Low',
     'Likely adjustment entries — no documentation on expected behaviour'),
    ('CREDIT_CHECK_DONE encoding',
     'Opaque codes',
     'Low',
     'Values (Y LOAN, Y CASH, Z DEMO) not documented in definitions file'),
    ('No loan-level payment history',
     'Snapshot architecture',
     'High',
     'Cannot build repayment curves, detect missed-payment streaks, or do vintage analysis'),
]

lim_df = pd.DataFrame(limitations, columns=['Issue','Scale','Severity','Analytical Impact'])
print('=== Data Limitations Summary ===')
print(lim_df.to_string(index=False))

=== Data Limitations Summary ===
                             Issue                                  Scale Severity                                                                          Analytical Impact
PAYMENT_AMOUNT / ADJUSTMENT_AMOUNT                             95.6% null     High          Cannot analyse payment frequency, missed payment runs or payment behaviour trends
             DOB & Income coverage                          ~50% of loans     High Age and income segmentation only possible for half the portfolio — biases segment analysis
          CUSTOMER_AGE column name                             Misleading   Medium  Name implies borrower age; actually loan tenure in days — high risk of misuse by analysts
                      NPS coverage                           17% of loans   Medium      Survey findings may not be representative of the full portfolio; response bias likely
           NPS duplicate responses 597 loans, 321 with conflicting scores   Medium           No d

## Q3.3 Recommendations

---

### Recommendation 1: Add a Loan-Level Payment Events Table

**Problem:** The current snapshot architecture captures balance positions at five points in time. PAYMENT_AMOUNT and ADJUSTMENT_AMOUNT are 95.6% null — populated only on the exact date a payment is made, so they are almost never captured in snapshots. There is no way to reconstruct how many payments a customer missed, when their last payment was, or whether they have a pattern of partial payments.

**Improvement:** Create a separate payment_events table with one row per payment event:

| Column | Type | Description |
|---|---|---|
| `loan_id` | string | Foreign key to loan |
| `event_date` | date | Date payment was received |
| `payment_amount` | decimal | Amount received |
| `expected_amount` | decimal | Amount due for that period |
| `payment_type` | enum | `regular`, `prepayment`, `adjustment`, `deposit` |
| `days_late` | integer | Days after due date (0 = on time) |
| `channel` | string | Payment channel (MoApp, M-Pesa, bank) |

**Why it matters:** Enables missed-payment streak detection (trigger collections at 2nd miss, not after 30 DPD), repayment curve analysis, vintage cohort tracking, and channel-level reconciliation — directly addressing the wrongful locking issue identified in Q2.

---

### Recommendation 2: Standardise and Mandate Demographic Capture at Onboarding

**Problem:** DOB and income data exists for only ~50% of loans because bureau checks are currently only run for credit-financed sales. Cash sales and legacy accounts have no demographic record. This makes risk segmentation unreliable — any insight about age or income bands is based on a potentially biased half of the portfolio.

**Improvement:** Make demographic fields mandatory at point of sale for all sale types:

| Field | Source | Currently Captured? |
|---|---|---|
| Date of birth | Customer-provided + ID verification | Credit sales only |
| National ID number | ID scan | Partial |
| Self-reported monthly income | Onboarding form | Credit sales only |
| Employment type | Onboarding form | Not captured |
| Gender | TransUnion bureau | Credit sales only |

Rename CUSTOMER_AGE → LOAN_TENURE_DAYS in the credit table to eliminate the naming ambiguity. Add a separate customer_age_at_sale field derived from DOB at origination date.

**Why it matters:** Full demographic coverage enables reliable risk segmentation, fairer credit scoring, better product targeting (e.g. 18–25 cohort needs different affordability terms), and compliance with credit reporting standards.

---

### Recommendation 3: Implement a Single NPS Response Per Loan with Timestamped History

**Problem:** 597 loans have multiple NPS responses, of which 321 have conflicting scores. There is no rule for which response to use — most recent? highest? average? This makes loan-level satisfaction metrics ambiguous and can skew portfolio-level NPS calculations. Additionally, NPS is not linked to a snapshot date, so it is impossible to know whether a customer's satisfaction score reflects their experience at DPD 0 or DPD 90.

**Improvement:** Restructure NPS capture as a timestamped history table:

| Column | Type | Description |
|---|---|---|
| `loan_id` | string | Foreign key |
| `survey_date` | date | Date survey was submitted |
| `nps_score` | integer | 0–10 |
| `account_status_at_survey` | string | Status snapshotted at survey time |
| `dpd_at_survey` | integer | DPD snapshotted at survey time |
| `trigger_event` | enum | `post_sale`, `post_payment`, `post_lock`, `post_resolution` |

Apply a deduplication rule for reporting: use the **most recent response** per loan per calendar quarter.

**Why it matters:** Linking NPS to the credit state at the time of survey makes satisfaction analysis causal rather than correlational — you can directly measure the NPS impact of a locking event, a missed payment, or a support interaction, enabling closed-loop feedback between the collections and customer experience teams.

##  Entity-Relationship Diagram

The diagram below shows the proposed table structure. Key design decisions:
- loan is the central fact — every other table keys off loan_id
- payment_event is a **new** table (currently missing — Q3 Recommendation 1)
- nps_response links to loan via loan_id and carries snapshot_date at time of survey (Q3 Recommendation 3)
- customer holds one row per person; loan holds one row per loan — a customer can have multiple loans

In [63]:
print("""
                    ┌──────────────────────────────┐
                    │         dim_customer         │
                    ├──────────────────────────────┤
                    │ PK  customer_id             │
                    │     date_of_birth           │
                    │     gender                  │
                    │     citizenship             │
                    │     avg_monthly_income      │
                    │     income_band             │
                    │     employment_type         │
                    └──────────────┬──────────────┘
                                   │ 1 : N
                                   ▼
                    ┌──────────────────────────────┐
                    │          dim_loan            │
                    ├──────────────────────────────┤
                    │ PK  loan_id                 │
                    │ FK  customer_id            │
                    │     sale_date              │
                    │     sale_type              │
                    │     product_name           │
                    │     cash_price             │
                    │     loan_price             │
                    │     deposit                │
                    │     weekly_rate            │
                    │     loan_term              │
                    │     credit_check_done      │
                    │     credit_expiry          │
                    └──────────────┬──────────────┘
                                   │ 1 : N
        ┌──────────────────────────┼──────────────────────────┐
        ▼                          ▼                          ▼

┌──────────────────────────┐  ┌──────────────────────────┐  ┌──────────────────────────┐
│  fct_credit_snapshot     │  │     fct_payment_event    │  │     fct_nps_response     │
├──────────────────────────┤  ├──────────────────────────┤  ├──────────────────────────┤
│ PK  loan_id              │  │ PK  event_id             │  │ PK  response_id          │
│ PK  snapshot_date        │  │ FK  loan_id              │  │ FK  loan_id              │
│ FK  loan_id              │  │     payment_date         │  │     survey_date          │
│     snapshot_date        │  │     amount_paid          │  │     nps_score            │
│     balance              │  │     amount_expected      │  │     nps_category         │
│     closing_balance      │  │     payment_type         │  │     account_status_at_survey │
│     arrears              │  │     days_late            │  │     dpd_at_survey        │
│     days_past_due        │  │     payment_channel      │  │     trigger_event        │
│     total_paid           │  │                          │  │     locked_despite_payment │
│     total_due_today      │  │                          │  │     payment_delay_experienced │
│     balance_due_status   │  │                          │  │     support_difficulty   │
│     account_status_l1    │  │                          │  └──────────────────────────┘
│     account_status_l2    │  │                          │
└──────────────────────────┘  └──────────────────────────┘

""")



                    ┌──────────────────────────────┐
                    │         dim_customer         │
                    ├──────────────────────────────┤
                    │ PK  customer_id             │
                    │     date_of_birth           │
                    │     gender                  │
                    │     citizenship             │
                    │     avg_monthly_income      │
                    │     income_band             │
                    │     employment_type         │
                    └──────────────┬──────────────┘
                                   │ 1 : N
                                   ▼
                    ┌──────────────────────────────┐
                    │          dim_loan            │
                    ├──────────────────────────────┤
                    │ PK  loan_id                 │
                    │ FK  customer_id            │
                    │     sale_date              │
                    │     sale_